# 5G RCF Anomaly Detection Demo

This notebook demonstrates **AWS Managed Prometheus RCF anomaly detection** on a live 5G network.

**Scenario**: A bad config push to AMF1 causes ~50 users to lose registration. RCF detects the anomaly instantly. The DevOps Agent correlates with infrastructure metrics to identify root cause.

---

## Architecture

![Architecture](architecture.png)

## RCF Data Flow

![RCF Dataflow](rcf-dataflow.png)

## Demo Scenario (Before → Fault → Recovery)

![Fault Scenario](fault-scenario.png)

---

## Setup

Installs kubectl (if missing), configures kubeconfig, and verifies connectivity to the EKS cluster.
All kubectl calls use a timeout so the notebook never hangs silently.

In [ ]:
import boto3
import json
import time
import subprocess
import os
from botocore.auth import SigV4Auth
from botocore.awsrequest import AWSRequest
import requests

# Configuration
REGION = 'us-east-1'
WORKSPACE_ID = 'ws-185ff7f8-c698-4d0e-9135-945b03aeccd1'
AMP_QUERY_URL = f'https://aps-workspaces.{REGION}.amazonaws.com/workspaces/{WORKSPACE_ID}/api/v1/query'
EKS_CLUSTER = 'open5gs-amp-cluster'

session = boto3.Session(region_name=REGION)
credentials = session.get_credentials().get_frozen_credentials()

def query_amp(promql):
    """Execute a PromQL query against AMP with SigV4 auth."""
    params = {'query': promql}
    req = AWSRequest(method='POST', url=AMP_QUERY_URL, data=params)
    SigV4Auth(credentials, 'aps', REGION).add_auth(req)
    resp = requests.post(AMP_QUERY_URL, data=params, headers=dict(req.headers), timeout=30)
    return resp.json()['data']['result']

def run_kubectl(args, stdin=None, timeout=60):
    """Run a kubectl command with a timeout. Returns (ok, stdout, stderr).
    Never hangs: raises a clear message on timeout."""
    try:
        r = subprocess.run(['kubectl'] + args, input=stdin,
                           capture_output=True, text=True, timeout=timeout)
        if r.returncode != 0:
            print(f'  ⚠ kubectl {" ".join(args[:2])} failed (rc={r.returncode}):')
            print(f'    {r.stderr.strip()[:300]}')
        return (r.returncode == 0, r.stdout, r.stderr)
    except subprocess.TimeoutExpired:
        print(f'  ⚠ kubectl {" ".join(args[:2])} TIMED OUT after {timeout}s')
        print('    → Likely cause: notebook IAM role lacks EKS access, or endpoint unreachable.')
        print('    → Fix: aws eks create-access-entry + associate-access-policy for the notebook role.')
        return (False, '', 'timeout')

# Install kubectl if missing
if not os.path.exists('/usr/local/bin/kubectl'):
    print('Installing kubectl...')
    subprocess.run(['curl', '-sLO', 'https://dl.k8s.io/release/v1.31.0/bin/linux/amd64/kubectl'], capture_output=True, timeout=120)
    subprocess.run(['chmod', '+x', 'kubectl'], capture_output=True)
    subprocess.run(['sudo', 'mv', 'kubectl', '/usr/local/bin/'], capture_output=True)

# Configure kubeconfig
subprocess.run(['aws', 'eks', 'update-kubeconfig', '--region', REGION, '--name', EKS_CLUSTER],
               capture_output=True, timeout=60)

print(f'✓ AMP workspace: {WORKSPACE_ID}')
print(f'  Region: {REGION}  |  EKS: {EKS_CLUSTER}')

# Connectivity check (5s timeout - fails fast if auth/network broken)
ok, out, err = run_kubectl(['get', 'nodes', '--no-headers'], timeout=15)
if ok:
    print(f'✓ kubectl connected: {len(out.strip().splitlines())} nodes ready')
else:
    print('✗ kubectl NOT connected — fix EKS access before running fault injection cells.')

## Step 1: Verify Baseline (Healthy State)

All 100 UEs should be registered across 2 AMFs. RCF score should be 0.

In [ ]:
results = query_amp('fivegs_amffunction_rm_registeredsubnbr')
print('═══ Registered Subscribers (per AMF) ═══')
total = 0
for r in results:
    pod = r['metric'].get('pod', 'unknown')
    value = int(r['value'][1])
    total += value
    print(f'  {pod}: {value} UEs')
print(f'  ─────────────────────')
print(f'  TOTAL: {total} UEs registered')
print()
rcf_metrics = query_amp('{__name__=~"anomaly_detector:.+", alias="5g-registered-subscribers"}')
print('═══ RCF Anomaly Detector ═══')
for r in rcf_metrics:
    name = r['metric']['__name__'].replace('anomaly_detector:', '')
    print(f'  {name:12s}: {r["value"][1]}')
print()
print('✓ Baseline healthy' if total >= 80 else '⚠ Subscribers below expected')

## Step 2: Inject Fault (Bad Config Push to AMF1)

Pushes a broken config (missing required `time.t3512`) to AMF1. It crashes → CrashLoopBackOff → ~50 UEs deregister. AMF2 is unaffected.

In [ ]:
broken_config = '''sbi:\n  server:\n    no_tls: true\n  client:\n    no_tls: true\namf:\n  sbi:\n    - addr: 0.0.0.0\n      port: 7777\n  ngap:\n    - addr: 0.0.0.0\n  metrics:\n    - addr: 0.0.0.0\n      port: 9090\n  guami:\n    - plmn_id: {mcc: 999, mnc: 70}\n      amf_id: {region: 2, set: 1}\n  tai:\n    - plmn_id: {mcc: 999, mnc: 70}\n      tac: 1\n  plmn_support:\n    - plmn_id: {mcc: 999, mnc: 70}\n      s_nssai:\n        - sst: 1\n  security:\n    integrity_order: [NIA2, NIA1, NIA0]\n    ciphering_order: [NEA0, NEA1, NEA2]\n  network_name:\n    full: Open5GS\n  amf_name: open5gs-amf1\nscp:\n  sbi:\n    - addr: scp.open5gs.svc.cluster.local\n      port: 7777\n'''

print('═══ FAULT INJECTION ═══')
print('Pushing broken config to AMF1 (missing time.t3512)...')

# Generate configmap YAML (client-side, instant)
ok, cm_yaml, _ = run_kubectl(
    ['create', 'configmap', 'amf1-config', '-n', 'open5gs',
     f'--from-literal=amf.yaml={broken_config}', '--dry-run=client', '-o', 'yaml'],
    timeout=15)

if ok:
    # Apply it (server-side, 30s timeout)
    ok2, _, _ = run_kubectl(['apply', '-f', '-'], stdin=cm_yaml, timeout=30)
    if ok2:
        # Restart AMF1
        run_kubectl(['rollout', 'restart', 'deploy/amf1', '-n', 'open5gs'], timeout=30)
        print('✗ Bad config pushed. AMF1 will enter CrashLoopBackOff.')
        print('  ~50 UEs will lose registration within 30s.')
        print()
        print('Waiting 45s for impact to propagate to AMP...')
        time.sleep(45)
    else:
        print('✗ Could not apply config — check kubectl connectivity (Setup cell).')
else:
    print('✗ Could not generate config — check kubectl is installed (Setup cell).')

## Step 3: Observe the Anomaly

In [ ]:
results = query_amp('fivegs_amffunction_rm_registeredsubnbr')
print('═══ AFTER FAULT: Registered Subscribers ═══')
total = 0
for r in results:
    pod = r['metric'].get('pod', 'unknown')
    value = int(r['value'][1])
    total += value
    status = '✗ DOWN' if 'amf1' in pod and value == 0 else '✓ OK'
    print(f'  {pod}: {value} UEs  {status}')
print(f'  TOTAL: {total} UEs (was 100)  |  IMPACT: {100 - total} users lost service')
print()
rcf_metrics = query_amp('{__name__=~"anomaly_detector:.+", alias="5g-registered-subscribers"}')
print('═══ RCF Anomaly Detector ═══')
for r in rcf_metrics:
    name = r['metric']['__name__'].replace('anomaly_detector:', '')
    ind = ' ← ANOMALY!' if name == 'score' and float(r['value'][1]) > 0 else ''
    print(f'  {name:12s}: {r["value"][1]}{ind}')

## Step 4: Root Cause Analysis

In [ ]:
print('═══ ROOT CAUSE ANALYSIS ═══\n')
print('1. Per-AMF breakdown:')
for r in query_amp('fivegs_amffunction_rm_registeredsubnbr'):
    pod = r['metric'].get('pod', '')
    v = r['value'][1]
    print(f'   {pod}: {v} registered', '← AFFECTED' if int(v) == 0 else '')
print()
print('2. Pod restart count:')
for r in query_amp('kube_pod_container_status_restarts_total{namespace="open5gs", pod=~"amf.*"}'):
    pod = r['metric'].get('pod', '')
    rst = r['value'][1]
    print(f'   {pod}: {rst} restarts', '← CRASH LOOP!' if int(float(rst)) > 2 else '')
print()
print('3. Node health:')
nodes = query_amp('kube_node_spec_unschedulable')
print('   All nodes schedulable ✓ (not a node issue)' if not nodes else f'   {len(nodes)} node(s) unschedulable')
print()
print('═══ CONCLUSION ═══')
print('Root cause: AMF1 in CrashLoopBackOff after a config change.')
print('Impact: ~50 users on TAC=1 lost registration. AMF2 healthy.')
print('Recommendation: Rollback amf1-config ConfigMap.')

## Step 5: Recovery

In [ ]:
valid_config = '''sbi:\n  server:\n    no_tls: true\n  client:\n    no_tls: true\n\ntime:\n  t3512:\n    value: 540\n\namf:\n  sbi:\n    - addr: 0.0.0.0\n      port: 7777\n  ngap:\n    - addr: 0.0.0.0\n  metrics:\n    - addr: 0.0.0.0\n      port: 9090\n  guami:\n    - plmn_id: {mcc: 999, mnc: 70}\n      amf_id: {region: 2, set: 1}\n  tai:\n    - plmn_id: {mcc: 999, mnc: 70}\n      tac: 1\n  plmn_support:\n    - plmn_id: {mcc: 999, mnc: 70}\n      s_nssai:\n        - sst: 1\n  security:\n    integrity_order: [NIA2, NIA1, NIA0]\n    ciphering_order: [NEA0, NEA1, NEA2]\n  network_name:\n    full: Open5GS\n  amf_name: open5gs-amf1\nscp:\n  sbi:\n    - addr: scp.open5gs.svc.cluster.local\n      port: 7777\n'''

print('═══ RECOVERY ═══')
print('Restoring valid config to AMF1...')
ok, cm_yaml, _ = run_kubectl(
    ['create', 'configmap', 'amf1-config', '-n', 'open5gs',
     f'--from-literal=amf.yaml={valid_config}', '--dry-run=client', '-o', 'yaml'],
    timeout=15)
if ok:
    run_kubectl(['apply', '-f', '-'], stdin=cm_yaml, timeout=30)
    run_kubectl(['rollout', 'restart', 'deploy/amf1', '-n', 'open5gs'], timeout=30)
    print('✓ Valid config restored. Waiting 60s for UEs to re-register...')
    time.sleep(60)
    results = query_amp('sum(fivegs_amffunction_rm_registeredsubnbr)')
    total = int(results[0]['value'][1]) if results else 0
    print(f'\n═══ POST-RECOVERY ═══')
    print(f'  Registered subscribers: {total}')
    print(f'  Status: {"✓ RECOVERED" if total >= 80 else "⏳ Still recovering..."}')

---

## Summary

| Phase | registeredsubnbr | RCF Score | Root Cause |
|---|---|---|---|
| Healthy | 100 | 0 | — |
| Fault injected | ~50 | >0.1 | AMF1 CrashLoopBackOff (bad config) |
| Recovered | 100 | 0 | Config rolled back |

### Key Takeaways
1. **RCF detects onset** — the moment subscribers drop, score spikes
2. **Infra correlation** — pod restarts + per-AMF breakdown pinpoints root cause
3. **Partial impact** — only TAC=1 users affected; TAC=2 is healthy
4. **Fast recovery** — fix config, restart, UEs auto-re-register